# cleaning 

In [ ]:
# importing necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import pymongo
from pymongo import MongoClient, errors
import glob 
import json

In [ ]:
# initialising lists of years we iterate through 
months_years = ["01_2024", "04_2024", "07_2024", "10_2024", 
                "02_2025", "04_2025", "08_2025", "11_2025", 
                "02_2026", "05_2026"] 
years_1 = ["2021", "2022", "2023", "2024", "2025", "2026"] 
years_2 = ["2021", "2022", "2023", "2024"]

In [ ]:
# initalising mongo connection string
client = MongoClient("mongodb+srv://<username>:<password>@ds4320proj2.p0udp7x.mongodb.net/")
database = client["DSProj2"]

### unplanned visits 

In [ ]:
# turn city/town columns into 'city' columns 
unplanned_columns = ["Facility ID", "City", "Facility Name", 
                    "ZIP Code", "Measure ID", "Start Date", "End Date"]

unplanned_all = pd.DataFrame(unplanned_columns) 

for month_year in months_years: 
    unplanned_current = pd.read_csv(f"./hospital_data/unplanned_visits/{month_year}_unplanned_visits.csv", low_memory=False)
    if "City/Town" in list(unplanned_current.columns): 
        unplanned_current["City"] = unplanned_current["City/Town"]
    unplanned4final = unplanned_current[unplanned_columns]
    unplanned_all = pd.concat([unplanned_all, unplanned4final], ignore_index=True) 


In [ ]:
# unplanned_all.info(verbose=True)

In [ ]:
# prettifying spreadsheet 
unplanned_all = unplanned_all.drop([0], axis=1)
unplanned_all = unplanned_all.drop(list(range(0,8)), axis=0)
unplanned_all["ZIP"] = unplanned_all["ZIP Code"]
unplanned_all = unplanned_all.drop(["ZIP Code"], axis=1)

In [ ]:
# converting data types 
unplanned_all["Start Date"] = pd.to_datetime(unplanned_all["Start Date"], format="%m/%d/%Y")
unplanned_all["End Date"] = pd.to_datetime(unplanned_all["End Date"], format="%m/%d/%Y") 

In [ ]:
# double-checking our data conversion 
# unplanned_all.info()

In [ ]:
# slide the data into mongo 
unplanned_docs = database["unplanned_data"]
unplanned_docs.insert_many(unplanned_all.to_dict("records"));

### readmission reduction data

In [ ]:
# readmission reduction data
readm_columns = ["Facility ID", "Facility Name", "Measure Name", "Start Date", "End Date", 
                 "Excess Readmission Ratio"]

readm_all = pd.DataFrame(readm_columns) 

# read in readmin data
for month_year in months_years: 
    readm_current = pd.read_csv(f"./hospital_data/readmission_reduction/{month_year}_readm_reduc.csv")
    readm4final = readm_current[readm_columns]
    readm_all = pd.concat([readm_all, readm4final], ignore_index=True)

In [ ]:
# readm_all.info(verbose=True)

In [ ]:
# rename column 
readm_all["Measure ID"] = readm_all["Measure Name"] 
readm_all = readm_all.drop(["Measure Name"], axis=1)

In [ ]:
# converting to datetime
readm_all["Start Date"] = pd.to_datetime(readm_all["Start Date"], format="%m/%d/%Y")
readm_all["End Date"] = pd.to_datetime(readm_all["End Date"], format="%m/%d/%Y")
# readm_all["End Date"].head(20)

In [ ]:
# start date / end date EDA 
# plt.hist(readm_all["Start Date"]) 
# plt.hist(readm_all["End Date"]) 
# plt.show()

In [ ]:
# slide the data into mongo 
readm_docs = database["readm_data"]

readm_docs.insert_many(readm_all.to_dict("records"));

### generator data

In [ ]:
# loop through each, slide the data into a dataframe 
generators_columns = ["Plant Name", "County", "State", "Technology"]

generators_all = pd.DataFrame(generators_columns)

for year in years_2: 
    generator_current = pd.read_csv(f"./plant_data/generator_details/eia860_generator_{year}.csv", low_memory=False)
    generator4final = generator_current[generators_columns]
    generators_all = pd.concat([generators_all, generator4final], ignore_index=True)

# generators_all.info(verbose=True)

In [ ]:
# make prettier 
generators_all = generators_all.drop([0], axis=1)
generators_all = generators_all.drop(list(range(0,6)), axis=0)
# generators_all.head()

In [ ]:
# slide the data into mongo 
generators_docs = database["generators_data"]

generators_docs.insert_many(generators_all.to_dict("records"));

### power plant data

In [ ]:
# initialise our dataframe 
plants_columns = ["Plant Name", "County", "State", "Zip"]

plants_all = pd.DataFrame(plants_columns)

for year in years_2:
    plant_current = pd.read_csv(f"./plant_data/plant_details/eia860_plant_{year}.csv")
    plant4final = plant_current[plants_columns]
    plants_all = pd.concat([plants_all, plant4final], ignore_index=True)

# plants_all.info(verbose=True)

In [ ]:
# drop the first 6 columns. 
plants_all = plants_all.drop([0], axis=1)
plants_all = plants_all.drop(list(range(0,6)), axis=0)
plants_all["Zip"] = pd.to_numeric(plants_all["Zip"], errors="coerce")
plants_all["ZIP"] = plants_all["Zip"]
plants_all = plants_all.drop(["Zip"], axis=1)
plants_all.head()

In [ ]:
# slide the data into mongo 
plants_docs = database["plants_data"]

plants_docs.insert_many(plants_all.to_dict("records"));

### demand data

In [ ]:
# initialise dataframe 
demand_columns = ["Plant Name", "Elec_MMBtu January", "Elec_MMBtu February","Elec_MMBtu March", 
                  "Elec_MMBtu April", "Elec_MMBtu May", "Elec_MMBtu June", 
                  "Elec_MMBtu July", "Elec_MMBtu August", "Elec_MMBtu September", 
                  "Elec_MMBtu October", "Elec_MMBtu November", "Elec_MMBtu December", 
                 "YEAR"]
                  
demand_all = pd.DataFrame(demand_columns)   

for year in years_1:
    demand_current = pd.read_csv(f"./demand_data/eia923_demand_{year}.csv", low_memory=False)
    demand_current.columns = (
        demand_current.columns
        .str.replace("\n", " ", regex=False)
        .str.strip()
    )
    demand4final = demand_current[demand_columns]
    demand_all = pd.concat([demand_all, demand4final], ignore_index=True)


# demand_all.info(verbose=True)

In [ ]:
# converting our months data to months 
demand_months = ["Elec_MMBtu January", "Elec_MMBtu February","Elec_MMBtu March", 
                  "Elec_MMBtu April", "Elec_MMBtu May", "Elec_MMBtu June", 
                  "Elec_MMBtu July", "Elec_MMBtu August", "Elec_MMBtu September", 
                  "Elec_MMBtu October", "Elec_MMBtu November", "Elec_MMBtu December"]

for month in demand_months: 
    demand_all[month] = pd.to_numeric(demand_all[month], errors="coerce") 

# demand_all.info(verbose=True)


In [ ]:
# drop the first 17 columns. 
demand_all = demand_all.drop(list(range(0,17)), axis=0).reset_index()
demand_all = demand_all.drop([0], axis=1)
demand_all = demand_all.drop(["index"], axis=1)
demand_all.head()

In [ ]:
# slide the data into mongo 
demand_docs = database["demand_data"]

demand_docs.insert_many(demand_all.to_dict("records"));


# pre-analysis

In [ ]:
# i'm putting this here just in case 
import pandas as pd
from pandas import DataFrame
import numpy as np
import matplotlib.pyplot as plt 
import pymongo
from pymongo import MongoClient, errors
import glob 
import json

In [ ]:
# turning those collections into dictionaries again 
unplanned_all = pd.DataFrame(list(database.unplanned_data.find({})))
readm_all = pd.DataFrame(list(database.readm_data.find({})))
plants_all = pd.DataFrame(list(database.plants_data.find({})))
generators_all = pd.DataFrame(list(database.generators_data.find({})))
demand_all = pd.DataFrame(list(database.demand_data.find({})))

In [ ]:
# investigating schema 
# unplanned_all.info()
# demand_all.info()
# plants_all.info()
# generators_all.info()
# demand_all.info()

In [ ]:
# removing the _id columns generated by mongo for our own analysis  
unplanned_all = unplanned_all.drop(["_id"], axis=1) 
readm_all = readm_all.drop(["_id"], axis=1)
plants_all = plants_all.drop(["_id"], axis=1)
generators_all = generators_all.drop(["_id"], axis=1)
demand_all = demand_all.drop(["_id"], axis=1) 

In [ ]:
# average excess readm rate per hospital 
readm_per_hospital = readm_all.merge(
    unplanned_all,
    on="Facility Name",
    how="left"
)
# readm_per_hospital.head()

In [ ]:
# average excess readm per zip code 
ex_readm_avg = (
    zip_per_hospital
    .groupby("ZIP", as_index=False)["Excess Readmission Ratio"]
    .mean()
    .rename(columns={"Excess Readmission Ratio": "Excess_Readm_Avg"})
)

In [ ]:
# one hot encoder for plant data 
encoded_technology = pd.get_dummies(generators_all["Technology"], dtype=int)
# surprise tool to help us later
encoded_tech_columns = list(encoded_technology.columns)

In [ ]:
# Displaying the encoded DataFrame
# encoded_technology.head()

In [ ]:
# dropping technology column from generators data so we don't have repetiton 
generators_all = generators_all.drop(["Technology"], axis=1)
# generators_all.head()

In [ ]:
# average monthly demand per plant 
demand_no_only = demand_all.groupby("Plant Name").mean()

In [ ]:
# average monthly demand per plant DATAFRAME 
plants_and_demand = plants_all.merge(
    demand_no_only,
    on="Plant Name",
    how="left"
)
# plants_and_demand.head()

In [ ]:
# dropping state data because i realised i dont need it
plants_and_demand = plants_and_demand.drop(["State"], axis=1) 
# plants_and_demand.head()

In [ ]:
# making a comprehensive dataframe with the generator table and the 1hotencoder technology 
generators_and_tech = pd.concat([generators_all, encoded_technology], axis=1)
# generators_and_tech.head()

In [ ]:
# average 'type of generator' per plant 
plant_generators = (
    generators_and_tech.groupby('Plant Name')
      .agg({
          **{col: 'mean' for col in encoded_tech_columns},
          'County': 'first',
          'State': 'first',
      })
      .reset_index()
)
# plant_generators.head()

In [ ]:
# that previous dataframe, but with monthly demand per plant...
# ...concatenated to it 
plants_everything = plants_and_demand.merge(
    plant_generators,
    on="Plant Name",
    how="left"
)
# plants_everything.head()

In [ ]:
# concatenate excess readmission data to all generator data 
data_for_ml = plants_everything.merge(
    ex_readm_avg,
    on="ZIP",
    how="left"
)

In [ ]:
# call my mongo collection 
ml_docs = database["data_for_ml"]

# slide into my database 
ml_docs.insert_many(data_for_ml.to_dict("records"));

# machine learning

In [ ]:
# just in case 
import logging
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split
import pymongo
from pymongo import MongoClient, errors
import glob 
import json
from sklearn.model_selection import GridSearchCV 
from sklearn.model_selection import KFold
from sklearn.ensemble import HistGradientBoostingRegressor
import matplotlib.pyplot as plt
import shap

In [ ]:
# loading up all our data to use
ml_data = pd.DataFrame(list(database.data_for_ml.find({})))

In [ ]:
# taking a look at all of our data 
# ml_data.info()

In [ ]:
# feature matrix 
X_unprocessed = ml_data.iloc[:, 4:44]
# X_unprocessed.head()

In [ ]:
# target vector 
y_unprocessed = ml_data.iloc[:, 46]
# y_unprocessed.head()

In [ ]:
# im going to do a magic trick 
Xy = pd.concat([X_unprocessed, y_unprocessed], axis=1) 
# Xy.head()

In [ ]:
# interpolate data 
# to get rid of NaNs
Xy_int = Xy.interpolate()
# Xy_int.head()

In [ ]:
# interpolated dataframe 
Xy_int = Xy_int.dropna()
# Xy_int.info()

In [ ]:
# final feature matrix and target vector 
X = Xy_int.iloc[:, 0:39]
y = Xy_int.iloc[:, -1]

In [ ]:
# print(X.head())
# print(y.head())

In [ ]:
# train-test-split 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

In [ ]:
# model selection part 
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
# model initialisation 
hist_model = HistGradientBoostingRegressor(max_iter=2500, random_state=42)

In [ ]:
# fitting model 
hist_model.fit(X_train, y_train)

In [ ]:
# features for model selection 
max_depth_values = [5, 10, 15]

hist_param_grid = {
    'max_depth': max_depth_values,
    'learning_rate': [0.01, 0.1, 0.3]
}

hist_grid = GridSearchCV(hist_model, hist_param_grid, cv=cv,
                       scoring='neg_root_mean_squared_error', n_jobs=-1, verbose=1)

hist_grid.fit(X_train, y_train)

In [ ]:
# best parameters 
# print("Best parameters:", hist_grid.best_params_)
# print("Best CV score:", hist_grid.best_score_)

In [ ]:
best_model = hist_grid.best_estimator_

In [ ]:
# test score 
test_score = best_model.score(X_test, y_test)
# print("neg root mean squared error:", test_score)

In [ ]:
# calculating y_pred
y_pred = hist_grid.best_estimator_.predict(X_test)

In [ ]:
# viewing the most important features predicting excess readm 
explainer = shap.Explainer(best_model)
shap_values = explainer(X_test)

shap.plots.bar(shap_values)
plt.title("Most important features for predicting readm") 